# Business value: persistence, ARIMA(0,1,0), ARIMA(0,1,1), and drift

This report evaluates the fixed importer/exporter timing rule on equivalent origins and horizons. ARIMA(0,1,1) is a predefined alternative, not a model selected using test results. The USD amount is an explicit scenario input.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from trm_forecast.business.evaluation import evaluate_business_value, validate_forecast_grid
from trm_forecast.evaluation.arima import evaluate_arima
from trm_forecast.evaluation.persistence import evaluate_persistence
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data/interim/trm_normalized.csv').exists())
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
evaluation = config['evaluation']
data = pd.read_csv(ROOT / 'data/interim/trm_normalized.csv', parse_dates=['date'])
series = data.set_index('date')['trm'].sort_index()
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
protocol = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs)
result = protocol.split(series)
horizons = tuple(evaluation['horizons'])
usd_amount = 100_000  # scenario input; replace with the operation's amount
arima_kwargs = dict(min_history=config['arima']['min_history'], maxiter=config['arima']['maxiter'], refit_frequency=config['arima']['refit_frequency'])

In [ ]:
# Generate all four level-forecast sets with the existing temporal evaluators.
persistence, persistence_metrics = evaluate_persistence(result, horizons)
persistence = persistence[persistence['partition'].isin(['validation', 'test'])].copy()
persistence['model'] = 'persistence_last'
arima010, arima010_metrics = evaluate_arima(result, (0, 1, 0), horizons, ('validation', 'test'), **arima_kwargs)
arima011, arima011_metrics = evaluate_arima(result, (0, 1, 1), horizons, ('validation', 'test'), **arima_kwargs)
arima010_drift, arima010_drift_metrics = evaluate_arima(result, (0, 1, 0), horizons, ('validation', 'test'), trend=config['arima']['drift_trend'], **arima_kwargs)

def successful_arima(frame):
    return frame[frame['status'].isin(['converged', 'converged_with_warning'])].copy()

forecasts = pd.concat([persistence, successful_arima(arima010), successful_arima(arima011), successful_arima(arima010_drift)], ignore_index=True)
forecasts['model'] = forecasts['model'].replace({'arima(0, 1, 0)': 'arima_010', 'arima(0, 1, 1)': 'arima_011', 'arima(0, 1, 0)_drift': 'arima_010_drift'})
forecasts.groupby(['partition', 'model']).size().rename('forecast_rows').reset_index()

In [ ]:
# Audit the drift forecast path at the same validation and test origins.
drift_valid = successful_arima(arima010_drift)
drift_audit_rows = []
for partition in ['validation', 'test']:
    origin = drift_valid.loc[drift_valid['partition'].eq(partition), 'forecast_origin'].min()
    group = drift_valid[(drift_valid['partition'] == partition) & (drift_valid['forecast_origin'] == origin)].sort_values('horizon')
    importer = group.loc[group['prediction'].idxmin()]
    exporter = group.loc[group['prediction'].idxmax()]
    for _, row in group.iterrows():
        drift_audit_rows.append({
            'model': 'arima_010_drift', 'partition': partition, 'forecast_origin': origin,
            'target_date': row['target_date'], 'horizon': int(row['horizon']),
            'prediction': row['prediction'], 'actual': row['actual'],
            'forecast_min': group['prediction'].min(), 'forecast_max': group['prediction'].max(),
            'importer_selected_horizon': int(importer['horizon']),
            'exporter_selected_horizon': int(exporter['horizon']),
            'importer_realized_at_selection': importer['actual'],
            'exporter_realized_at_selection': exporter['actual'],
            'realized_at_horizon_1': group.iloc[0]['actual'],
        })
pd.DataFrame(drift_audit_rows)

In [ ]:
# Predictive metrics by partition and horizon; no test result is used for selection.
forecast_metrics = pd.concat([
    persistence_metrics.assign(model='persistence_last', failed_predictions=0),
    arima010_metrics.assign(model='arima_010'),
    arima011_metrics.assign(model='arima_011'),
    arima010_drift_metrics.assign(model='arima_010_drift'),
], ignore_index=True)
forecast_metrics = forecast_metrics[forecast_metrics['partition'].isin(['validation', 'test'])]
forecast_metrics[['partition', 'model', 'horizon', 'observations', 'failed_predictions', 'mae', 'rmse']].sort_values(['partition', 'horizon', 'model'])

In [ ]:
# Audit actual ARIMA(0,1,1) forecasts for one complete origin in each partition.
arima011_valid = successful_arima(arima011)
audit_rows = []
for partition in ['validation', 'test']:
    origin = arima011_valid.loc[arima011_valid['partition'].eq(partition), 'forecast_origin'].min()
    group = arima011_valid[(arima011_valid['partition'] == partition) & (arima011_valid['forecast_origin'] == origin)].sort_values('horizon')
    importer = group.loc[group['prediction'].idxmin()]
    exporter = group.loc[group['prediction'].idxmax()]
    for _, row in group.iterrows():
        audit_rows.append({
            'partition': partition, 'forecast_origin': origin, 'target_date': row['target_date'],
            'horizon': int(row['horizon']), 'prediction': row['prediction'], 'actual': row['actual'],
            'forecast_min': group['prediction'].min(), 'forecast_max': group['prediction'].max(),
            'importer_selected_horizon': int(importer['horizon']),
            'exporter_selected_horizon': int(exporter['horizon']),
            'importer_realized_at_selection': importer['actual'],
            'exporter_realized_at_selection': exporter['actual'],
            'realized_at_horizon_1': group.iloc[0]['actual'],
        })
pd.DataFrame(audit_rows)

In [ ]:
# Report incomplete, duplicate, and invalid origin/model groups before comparison.
issues = validate_forecast_grid(forecasts, horizons, strict=False)
if issues.empty:
    exclusions = pd.DataFrame(columns=['partition', 'model', 'issue', 'excluded_origin_model_groups', 'excluded_operation_decisions'])
else:
    issues['partition'] = issues['_partition']
    exclusions = (issues.drop_duplicates(['_partition', 'forecast_origin', 'model', 'issue'])
        .groupby(['partition', 'model', 'issue'], as_index=False)
        .size().rename(columns={'size': 'excluded_origin_model_groups'}))
    exclusions['excluded_operation_decisions'] = exclusions['excluded_origin_model_groups'] * 2

# Use only origin/model groups complete for every model so the comparison is like-for-like.
complete_by_model = {}
for (partition, model), group in forecasts.groupby(['partition', 'model']):
    group_issues = validate_forecast_grid(group, horizons, strict=False)
    bad = set(group_issues['forecast_origin']) if not group_issues.empty else set()
    complete_by_model[(partition, model)] = {origin for origin in group['forecast_origin'].unique() if origin not in bad and group.loc[group['forecast_origin'] == origin, 'horizon'].nunique() == len(horizons)}
common_origins = {}
for partition in forecasts['partition'].unique():
    sets = [complete_by_model.get((partition, model), set()) for model in ['persistence_last', 'arima_010', 'arima_011', 'arima_010_drift']]
    common_origins[partition] = set.intersection(*sets) if all(sets) else set()
comparable = forecasts[forecasts.apply(lambda row: row['forecast_origin'] in common_origins.get(row['partition'], set()), axis=1)].copy()
exclusions

In [ ]:
# Apply the same economic rule to the common origin set.
decisions, summary = evaluate_business_value(comparable, usd_amount=usd_amount, horizons=horizons, strict=True)
later = (decisions.assign(later=decisions['selected_horizon'] > 1)
    .groupby(['partition', 'model', 'operation'], as_index=False)['later'].mean())
later['later_than_1_pct'] = later['later'] * 100
business_comparison = summary.merge(later.drop(columns='later'), on=['partition', 'model', 'operation'])
business_comparison.sort_values(['partition', 'operation', 'model'])

In [ ]:
# Selected-horizon distributions and auditable decision examples.
horizon_distribution = (decisions.groupby(['partition', 'model', 'operation', 'selected_horizon'])
    .size().rename('decisions').reset_index())
horizon_distribution['pct_of_decisions'] = (horizon_distribution['decisions'] /
    horizon_distribution.groupby(['partition', 'model', 'operation'])['decisions'].transform('sum') * 100)
examples = decisions[['partition', 'model', 'operation', 'forecast_origin', 'selected_horizon',
    'predicted_trm_at_selection', 'realized_trm_at_selection',
    'realized_trm_horizon_1', 'gross_cop_benefit']].sort_values(
    ['partition', 'model', 'operation', 'forecast_origin']).head(18)
display(horizon_distribution)
display(examples)

In [ ]:
# Verify the expected persistence equivalence of ARIMA(0,1,0).
persistence_check = persistence.merge(
    arima010[['partition', 'forecast_origin', 'target_date', 'horizon', 'prediction']],
    on=['partition', 'forecast_origin', 'target_date', 'horizon'], suffixes=('_persistence', '_arima010'))
persistence_check['absolute_difference'] = (persistence_check['prediction_persistence'] - persistence_check['prediction_arima010']).abs()
pd.DataFrame({'matched_forecasts': [len(persistence_check)], 'max_absolute_difference': [persistence_check['absolute_difference'].max()]})

In [ ]:
# Confirm whether ARIMA(0,1,1) actually varies by forecast horizon.
arima011_shape = (successful_arima(arima011).pivot_table(index=['partition', 'forecast_origin'], columns='horizon', values='prediction')
    .assign(horizon_range=lambda frame: frame.max(axis=1) - frame.min(axis=1))
    .groupby(level='partition')['horizon_range'].agg(['count', 'max', 'mean']))
arima011_shape

## Interpretation

Positive values are hypothetical gross COP improvements over executing at horizon 1, not realized savings or guaranteed trading gains. Validation may inform a future model or policy choice; the test results above are descriptive only after the forecast specifications and decision rule were fixed. Results exclude spreads, fees, taxes, hedging costs, liquidity, execution uncertainty, and the risk that a selected future forecast is unavailable.